# ☕ Pipeline ETL - Ventas Starbucks (Google Colab)
**Dataset:** [`guadalupesaraviaweht/ventas-starbucks`](https://www.kaggle.com/datasets/guadalupesaraviaweht/ventas-starbucks)  
**Curso:** Data Science II  
**Entregable:** Pre-Entrega 5 - Construcción de un Pipeline de Ingesta y Limpieza  

Este notebook descarga automáticamente el dataset oficial desde **Kaggle** usando `kagglehub` y ejecuta el **Pipeline ETL modular**.

## 1. Instalación e Importación de Librerías

In [ ]:
# Instalación de kagglehub, pandas y pyarrow
!pip install --quiet kagglehub pandas pyarrow

import os
import glob
import unicodedata
import pandas as pd
import kagglehub

print("✓ Librerías instaladas y cargadas correctamente. Versión de Pandas:", pd.__version__)

## 2. Descarga del Dataset desde Kaggle con `kagglehub`

In [ ]:
# Descarga automática de la última versión del dataset de Ventas Starbucks desde Kaggle
path = kagglehub.dataset_download("guadalupesaraviaweht/ventas-starbucks")
print("Ruta del dataset descargado por kagglehub:", path)

# Localizar el archivo CSV descargado
csv_files = glob.glob(os.path.join(path, "*.csv"))
if csv_files:
    input_csv = csv_files[0]
    print(f"✓ Archivo CSV detectado: '{input_csv}'")
else:
    raise FileNotFoundError("No se encontró un archivo .csv en la ruta descargada por kagglehub.")

os.makedirs("data/output", exist_ok=True)
output_parquet = "data/output/ventas_starbucks_limpio.parquet"

## 3. Definición Modular de Funciones ETL

In [ ]:
# === ETAPA 1: EXTRACCIÓN ===
def extraer_datos_csv(filepath: str) -> pd.DataFrame:
    """Extrae los datos desde el archivo CSV de origen."""
    if not os.path.exists(filepath):
        raise FileNotFoundError(f"No se encontró el archivo: {filepath}")
    print(f"[EXTRACCIÓN] Leyendo origen: '{filepath}'...")
    df = pd.read_csv(filepath)
    print(f"[EXTRACCIÓN] ✓ Ingesta exitosa: {len(df)} filas, {len(df.columns)} columnas.")
    return df

# === ETAPA 2: TRANSFORMACIÓN ===
def normalizar_nombres_columnas(df: pd.DataFrame) -> pd.DataFrame:
    """Normaliza encabezados a minúsculas, sin tildes y en formato snake_case."""
    df_res = df.copy()
    nuevas_cols = []
    for col in df_res.columns:
        col_norm = unicodedata.normalize('NFD', col).encode('ascii', 'ignore').decode("utf-8")
        col_clean = col_norm.strip().lower().replace(" ", "_")
        nuevas_cols.append(col_clean)
    df_res.columns = nuevas_cols
    print(f"[NORMALIZACIÓN] Columnas en snake_case: {list(df_res.columns)}")
    return df_res

def eliminar_duplicados(df: pd.DataFrame) -> pd.DataFrame:
    """Remueve filas totalmente duplicadas."""
    df_res = df.copy()
    cant_dup = df_res.duplicated().sum()
    print(f"[DUPLICADOS] Registros duplicados detectados: {cant_dup}")
    if cant_dup > 0:
        df_res = df_res.drop_duplicates()
        print(f"[DUPLICADOS] ✓ {cant_dup} duplicados eliminados.")
    else:
        print("[DUPLICADOS] ✓ No se encontraron registros duplicados.")
    return df_res

def gestionar_valores_nulos(df: pd.DataFrame) -> pd.DataFrame:
    """Gestiona nulos mediante imputación calculada y por etiqueta."""
    df_res = df.copy()
    # Imputación estructural en 'tamano' (productos que no son bebidas)
    if 'tamano' in df_res.columns:
        nulos_tam = df_res['tamano'].isnull().sum()
        if nulos_tam > 0:
            df_res['tamano'] = df_res['tamano'].fillna('No Aplica')
            print(f"  ✓ [IMPUTACIÓN ESTRUCTURAL] Imputados {nulos_tam} nulos en 'tamano' como 'No Aplica'.")
            
    # Imputación matemática de total_venta
    if 'total_venta' in df_res.columns and 'precio_unitario' in df_res.columns and 'cantidad' in df_res.columns:
        mask_nulo = df_res['total_venta'].isnull()
        if mask_nulo.sum() > 0:
            df_res.loc[mask_nulo, 'total_venta'] = df_res.loc[mask_nulo, 'precio_unitario'] * df_res.loc[mask_nulo, 'cantidad']
            print(f"  ✓ [IMPUTACIÓN CALCULADA] Imputados {mask_nulo.sum()} nulos en 'total_venta' mediante (precio * cantidad).")
    
    # Imputación categórica de respaldo
    for col in ['categoria_producto', 'metodo_de_pago', 'metodo_de_compra']:
        if col in df_res.columns and df_res[col].isnull().sum() > 0:
            cnt = df_res[col].isnull().sum()
            df_res[col] = df_res[col].fillna('No Especificado')
            print(f"  ✓ [IMPUTACIÓN CATEGÓRICA] Imputados {cnt} nulos en '{col}' como 'No Especificado'.")
        
    if 'fecha' in df_res.columns:
        df_res['fecha'] = pd.to_datetime(df_res['fecha'])
        
    return df_res

def aplicar_transformaciones(df: pd.DataFrame) -> pd.DataFrame:
    """Coordinador de transformaciones."""
    print("\n--- INICIO TRANSFORMACIÓN ---")
    df_norm = normalizar_nombres_columnas(df)
    df_dedup = eliminar_duplicados(df_norm)
    df_clean = gestionar_valores_nulos(df_dedup)
    print("--- FIN TRANSFORMACIÓN ---\n")
    return df_clean

# === ETAPA 3: CARGA Y JUSTIFICACIÓN ===
def cargar_datos_parquet(df: pd.DataFrame, output_path: str) -> None:
    """Exporta el resultado a un archivo Parquet con compresión Snappy."""
    print(f"[CARGA] Guardando resultado en '{output_path}'...")
    output_dir = os.path.dirname(output_path)
    if output_dir and not os.path.exists(output_dir):
        os.makedirs(output_dir, exist_ok=True)
    df.to_parquet(output_path, index=False, engine='pyarrow')
    kb = os.path.getsize(output_path) / 1024
    print(f"[CARGA] ✓ Archivo Parquet generado correctamente ({kb:.2f} KB).")

def ejecutar_pipeline(input_path: str, output_path: str) -> pd.DataFrame:
    """Ejecuta el Pipeline completo."""
    df_raw = extraer_datos_csv(input_path)
    df_clean = aplicar_transformaciones(df_raw)
    cargar_datos_parquet(df_clean, output_path)
    return df_clean

## 4. Ejecución del Pipeline ETL

In [ ]:
print("==================================================")
print(" PIPELINE ETL - VENTAS STARBUCKS (KAGGLEHUB DATASET)")
print("==================================================")

df_resultado = ejecutar_pipeline(input_csv, output_parquet)

print("==================================================")
print(f" ¡Pipeline finalizado con éxito!")
print(f" Total de registros limpios procesados: {len(df_resultado)}")
print("==================================================")

## 5. Inspección del Resultado y Verificación de Parquet

In [ ]:
# Lectura del archivo Parquet recién generado para validar integridades de tipos
df_parquet = pd.read_parquet(output_parquet)

print("📊 Vista previa del DataFrame limpio generado desde Parquet:")
display(df_parquet.head(10))

print("\n🔍 Información de tipos de datos y nulos conservados en Parquet:")
df_parquet.info()